# 04 — Playwright Code Generation

This notebook loads the mapped IR produced by Notebook 03 and generates a
Playwright Python test file.

Pipeline:
NL → IR → Mapped IR → Code → Execution → Analysis


In [35]:
from pathlib import Path
import json


## 1. Load Mapped IR


In [36]:
ir_path = Path("../sample-data/ir_examples/home_page_mapped_ir.json")

with open(ir_path, "r") as f:
    mapped_ir = json.load(f)

mapped_ir


{'test_name': 'render_app_test',
 'description': 'Test Name: Home Page Test\nGoal: Verify the home page loads and shows expected content.\nURL: https://ai-test-automation-demo.onrender.com/\n\nSteps:\n1. Navigate to the home page.\n2. Verify the page loads successfully.\n3. Verify the heading "Hello from your Flask Demo App!" is visible.\n4. Verify the "Login" link is visible.\n5. Click the "Login" link.\n6. Verify the browser navigates to /login.\n\nExpected Result:\nThe home page loads and displays the correct heading and login link.\n',
 'steps': [{'action': 'navigate',
   'target': 'url',
   'value': 'https://ai-test-automation-demo.onrender.com/',
   'selector': None},
  {'action': 'navigate',
   'target': 'url',
   'value': 'https://ai-test-automation-demo.onrender.com/login',
   'selector': None},
  {'action': 'click',
   'target': 'login_button',
   'selector': "button[type='submit']"}],
 'metadata': {'priority': 'high'}}

## 2. Playwright Code Templates

We define small helper functions to convert IR steps into Playwright code.


In [37]:
def generate_enter_text(step):
    return f'    await page.fill("{step["selector"]}", "{step["value"]}")'

def generate_click(step):
    return f'    await page.click("{step["selector"]}")'

def generate_navigate(step):
    return f'    await page.goto("{step["value"]}")'

def generate_assert_text(step):
    return (
        f'    await expect(page.locator("{step["selector"]}")).to_have_text("{step["value"]}")'
    )

def generate_assert_visible(step):
    return f'    await expect(page.locator("{step["selector"]}")).to_be_visible()'

# Optional — your app doesn’t use Enter-to-submit, but harmless to keep
def generate_press_enter(step):
    return f'    await page.press("{step["selector"]}", "Enter")'


## 3. Step Dispatcher

This maps IR actions → Playwright code generator functions.


In [38]:
ACTION_MAP = {
    "enter_text": generate_enter_text,
    "click": generate_click,
    "navigate": generate_navigate,
    "assert_text": generate_assert_text,
    "assert_visible": generate_assert_visible,

    # Optional — your app doesn’t use Enter-to-submit, but harmless to keep
    "press_enter": generate_press_enter,
}


## 4. Generate Playwright Test Code


In [39]:
def generate_test_code(ir):
    test_name = ir["test_name"]

    # Header: imports + async test + async with block
    header = [
        "from playwright.async_api import async_playwright, expect",
        "import asyncio",
        "",
        f"async def test_{test_name}():",
        "    async with async_playwright() as p:",
        "        browser = await p.chromium.launch(headless=False)",
        "        page = await browser.new_page()",
        ""
    ]

    # Body: generate Playwright code for each IR step
    body = []
    for step in ir["steps"]:
        action = step["action"]

        if action == "navigate":
            body.append(f'        await page.goto("{step["value"]}")')

        elif action == "enter_text":
            body.append(f'        await page.fill("{step["selector"]}", "{step["value"]}")')

        elif action == "click":
            body.append(f'        await page.click("{step["selector"]}")')

        elif action == "assert_text":
            body.append(
                f'        await expect(page.locator("{step["selector"]}")).to_have_text("{step["value"]}")'
            )

        elif action == "assert_visible":
            body.append(
                f'        await expect(page.locator("{step["selector"]}")).to_be_visible()'
            )

        elif action == "press_enter":  # optional, not used in your app
            body.append(f'        await page.press("{step["selector"]}", "Enter")')

        else:
            body.append(f"        # Unsupported action: {action}")

    # Footer: close browser + main entrypoint
    footer = [
        "",
        "        await browser.close()",
        "",
        "if __name__ == '__main__':",
        f"    asyncio.run(test_{test_name}())"
    ]

    return "\n".join(header + body + footer)


## 5. Render the Test Code


In [40]:
test_code = generate_test_code(mapped_ir)
print(test_code)


from playwright.async_api import async_playwright, expect
import asyncio

async def test_render_app_test():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()

        await page.goto("https://ai-test-automation-demo.onrender.com/")
        await page.goto("https://ai-test-automation-demo.onrender.com/login")
        await page.click("button[type='submit']")

        await browser.close()

if __name__ == '__main__':
    asyncio.run(test_render_app_test())


## 6. Save Test File


In [41]:
output_dir = Path("../generated-tests")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / f"{mapped_ir['test_name']}_test.py"

with open(output_path, "w") as f:
    f.write(test_code)

output_path


PosixPath('../generated-tests/render_app_test_test.py')